# MENG 417 — Homework #4 | Final professor submission

**Run all cells once.** This concise version trains the required **three-Dense-layer MNIST network** using the four original MNIST `.gz` archives, calculates the **real test accuracy and confusion matrix**, and creates one **submission PDF with the full source code appended**.

The PDF answers only the four items requested by the professor, without extended lessons or a cheat sheet.

**Do not submit an unexecuted notebook as the final report.** The professor requests the PDF produced by the last cell.

In [ ]:
# 1. Write the exact Python implementation.
from pathlib import Path
SOURCE = '"""MENG 417 - Homework 4. Original IDX MNIST; 3-Dense network; PDF report."""\nfrom pathlib import Path\nfrom urllib.request import urlretrieve\nimport gzip\nimport random\nimport struct\nimport textwrap\n\nimport numpy as np\nimport tensorflow as tf\nimport matplotlib.pyplot as plt\nfrom sklearn.metrics import confusion_matrix\nfrom sklearn.model_selection import train_test_split\n\nSEED = 417\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntf.keras.utils.set_random_seed(SEED)\nROOT = Path("MENG417_HW4_Results")\nROOT.mkdir(exist_ok=True)\nDATA = Path("mnist_raw")\nDATA.mkdir(exist_ok=True)\nFILES = [\n    "train-images-idx3-ubyte.gz",\n    "train-labels-idx1-ubyte.gz",\n    "t10k-images-idx3-ubyte.gz",\n    "t10k-labels-idx1-ubyte.gz",\n]\nBASE = "https://storage.googleapis.com/cvdf-datasets/mnist/"\nfor name in FILES:\n    destination = DATA / name\n    if not destination.exists():\n        print("Downloading original MNIST:", name, flush=True)\n        urlretrieve(BASE + name, destination)\n\n\ndef load_images(path):\n    with gzip.open(path, "rb") as f:\n        magic, count, rows, cols = struct.unpack(">IIII", f.read(16))\n        if magic != 2051 or (rows, cols) != (28, 28):\n            raise ValueError("Invalid IDX image header")\n        data = np.frombuffer(f.read(), dtype=np.uint8)\n    if data.size != count * rows * cols:\n        raise ValueError("Invalid IDX image payload")\n    return data.reshape(count, 784).copy()\n\n\ndef load_labels(path):\n    with gzip.open(path, "rb") as f:\n        magic, count = struct.unpack(">II", f.read(8))\n        if magic != 2049:\n            raise ValueError("Invalid IDX label header")\n        data = np.frombuffer(f.read(), dtype=np.uint8)\n    if data.size != count:\n        raise ValueError("Invalid IDX label payload")\n    return data.copy()\n\n\ntrain_x = load_images(DATA / FILES[0])\ntrain_y = load_labels(DATA / FILES[1])\ntest_x = load_images(DATA / FILES[2])\ntest_y = load_labels(DATA / FILES[3])\nassert train_x.shape == (60000, 784) and train_y.shape == (60000,)\nassert test_x.shape == (10000, 784) and test_y.shape == (10000,)\n\n# Use only original training data for fitting/validation; reserve official test.\nidx_fit, idx_val = train_test_split(\n    np.arange(60000), test_size=10000, stratify=train_y, random_state=SEED\n)\nx_fit = train_x[idx_fit].astype(np.float32) / 255.0\ny_fit = train_y[idx_fit].astype(np.int64)\nx_val = train_x[idx_val].astype(np.float32) / 255.0\ny_val = train_y[idx_val].astype(np.int64)\nx_test = test_x.astype(np.float32) / 255.0\ny_test = test_y.astype(np.int64)\n\nmodel = tf.keras.Sequential([\n    tf.keras.Input(shape=(784,)),\n    tf.keras.layers.Dense(128, activation="relu", kernel_initializer="he_normal"),\n    tf.keras.layers.Dense(64, activation="relu", kernel_initializer="he_normal"),\n    tf.keras.layers.Dense(10, activation="softmax"),\n])\nassert len(model.layers) == 3 and model.count_params() == 109386\n\nLEARNING_RATE = 0.001\nBATCH_SIZE = 128\nMAX_EPOCHS = 15\nPATIENCE = 3\nmodel.compile(\n    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),\n    loss=tf.keras.losses.SparseCategoricalCrossentropy(),\n    metrics=["accuracy"],\n)\nhistory = model.fit(\n    x_fit, y_fit, validation_data=(x_val, y_val),\n    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, verbose=2,\n    callbacks=[tf.keras.callbacks.EarlyStopping(\n        monitor="val_loss", patience=PATIENCE, restore_best_weights=True\n    ), tf.keras.callbacks.TerminateOnNaN()],\n)\n\n# Official test set is evaluated only after fitting and model selection.\ntest_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)\npredictions = np.argmax(model.predict(x_test, batch_size=512, verbose=0), axis=1)\ncm = confusion_matrix(y_test, predictions, labels=np.arange(10))\ncorrect = int(np.trace(cm))\nassert cm.shape == (10, 10) and cm.sum() == 10000\nassert correct == int(np.count_nonzero(predictions == y_test))\nassert np.isclose(test_acc, correct / 10000, atol=1e-4)\nassert np.all(cm.sum(axis=1) == np.bincount(y_test, minlength=10))\n\nnp.savetxt(ROOT / "confusion_matrix.csv", cm, fmt="%d", delimiter=",")\n\n# A compact, legible confusion-matrix figure for the PDF.\nfig, ax = plt.subplots(figsize=(7.0, 6.0))\nax.imshow(cm, cmap="Blues")\nax.set(xticks=np.arange(10), yticks=np.arange(10),\n       xlabel="Predicted digit", ylabel="Actual digit")\nfor r in range(10):\n    for c in range(10):\n        ax.text(c, r, str(cm[r, c]), ha="center", va="center", fontsize=7.8,\n                color="white" if cm[r, c] > cm.max()/2 else "black")\nfig.tight_layout()\nfig.savefig(ROOT / "confusion_matrix.png", dpi=180)\nplt.close(fig)\n\n# One PDF: four requested answers, real results, then full runnable source.\nfrom reportlab.lib import colors\nfrom reportlab.lib.pagesizes import A4\nfrom reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle\nfrom reportlab.lib.units import mm\nfrom reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,\n                                TableStyle, Image, Preformatted, PageBreak)\n\nstyles = getSampleStyleSheet()\nstyles.add(ParagraphStyle(name="HeadHW", parent=styles["Heading2"],\n                          fontSize=11, leading=14, spaceBefore=9, spaceAfter=4,\n                          textColor=colors.HexColor("#15365A")))\nstyles.add(ParagraphStyle(name="BodyHW", parent=styles["BodyText"],\n                          fontSize=9.1, leading=13, spaceAfter=5))\nstyles.add(ParagraphStyle(name="CodeHW", fontName="Courier", fontSize=7.2,\n                          leading=9.2, spaceAfter=0))\n\nreport = ROOT / "MENG417_HW4_Submit_To_Professor.pdf"\npdf = SimpleDocTemplate(str(report), pagesize=A4,\n                        leftMargin=17*mm, rightMargin=17*mm,\n                        topMargin=16*mm, bottomMargin=16*mm)\nstory = []\n\ndef paragraph(s, style="BodyHW"):\n    story.append(Paragraph(s, styles[style]))\n\ndef table(data, widths):\n    t = Table(data, colWidths=widths, repeatRows=1)\n    t.setStyle(TableStyle([\n        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E8EEF5")),\n        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),\n        ("GRID", (0, 0), (-1, -1), .3, colors.HexColor("#A9B7C5")),\n        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),\n        ("FONTSIZE", (0, 0), (-1, -1), 8.3),\n        ("TOPPADDING", (0, 0), (-1, -1), 5),\n        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),\n    ]))\n    story.append(t)\n    story.append(Spacer(1, 3*mm))\n\nparagraph("MENG 417 - Homework #4", "Title")\nparagraph("MNIST Digit Classification - Three Fully Connected Layers")\nparagraph("1. Testing accuracy and confusion matrix", "HeadHW")\nparagraph(f"Correct predictions: <b>{correct:,} / 10,000</b>. "\n          f"Testing accuracy: <b>{100*test_acc:.2f}%</b>. "\n          f"Test loss: <b>{test_loss:.5f}</b>.")\nparagraph("The confusion matrix below contains actual labels as rows and predicted "\n          "labels as columns. Its entries sum to 10,000.")\nstory.append(Image(str(ROOT / "confusion_matrix.png"), width=118*mm,\n                   height=101*mm))\n\nparagraph("2. Network structure", "HeadHW")\ntable([\n    ["Layer", "Neurons / activation", "Parameters"],\n    ["Input", "784 grayscale pixels", "0"],\n    ["Dense 1", "128, ReLU", "100,480"],\n    ["Dense 2", "64, ReLU", "8,256"],\n    ["Dense 3", "10, Softmax", "650"],\n    ["Total", "Exactly 3 Dense layers", "109,386"],\n], [42*mm, 80*mm, 45*mm])\n\nparagraph("3. Loss function", "HeadHW")\nparagraph("<b>Sparse categorical cross-entropy</b> with integer labels 0-9 "\n          "and Softmax probabilities for the ten digit classes. The loss "\n          "penalizes low probability assigned to the true class.")\n\nparagraph("4. Training parameters, optimizer and dropout", "HeadHW")\ntable([\n    ["Setting", "Value"],\n    ["Optimizer", "Adam"],\n    ["Initial learning rate", "0.001"],\n    ["Batch size", "128"],\n    ["Maximum epochs", "15"],\n    ["Epochs actually run", str(len(history.history["loss"]))],\n    ["Validation", "10,000 of the original training images"],\n    ["Early stopping", "Validation loss; patience 3; restore best weights"],\n    ["Initialization", "He normal in hidden layers"],\n    ["Input normalization", "Pixel values / 255"],\n    ["Dropout", "Not used"],\n], [62*mm, 105*mm])\nparagraph("Dropout was not used because it is optional in the assignment. "\n          "If introduced, it randomly disables part of the hidden activations "\n          "during training to reduce overfitting, but can increase underfitting "\n          "when applied too strongly. Early stopping monitors validation loss.")\n\nstory.append(PageBreak())\nparagraph("Appendix - Python source code", "HeadHW")\nparagraph("The following code loads the original four IDX archives, trains the "\n          "network and calculates the results shown above.")\nsource = Path(__file__).read_text(encoding="utf-8")\n# The appendix includes all experiment code, without the PDF-export utility.\nsource = source.split("# One PDF: four requested answers")[0].rstrip()\n# Preserve source indentation and paginate the runnable experiment code.\nfor start in range(0, len(source.splitlines()), 40):\n    block = "\\n".join(source.splitlines()[start:start+40])\n    story.append(Preformatted(block, styles["CodeHW"], maxLineLength=100))\n\ndef footer(canvas, doc):\n    canvas.saveState()\n    canvas.setFont("Helvetica", 8)\n    canvas.drawRightString(A4[0]-17*mm, 8*mm, f"Page {doc.page}")\n    canvas.restoreState()\n\npdf.build(story, onFirstPage=footer, onLaterPages=footer)\nprint("TEST ACCURACY:", f"{100*test_acc:.2f}%")\nprint("CONFUSION MATRIX:\\n", cm)\nprint("SUBMISSION PDF:", report.resolve())\n'
Path("MENG417_HW4_Submission.py").write_text(SOURCE, encoding="utf-8")
print("Source saved: MENG417_HW4_Submission.py")

In [ ]:
# 2. Execute: download original IDX MNIST files, train, test, build PDF.
# On a fresh Colab runtime, TensorFlow and scikit-learn are usually preinstalled.
import runpy
runpy.run_path("MENG417_HW4_Submission.py", run_name="__main__")

In [ ]:
# 3. Download the single PDF required by the professor.
from pathlib import Path
pdf_path = Path("MENG417_HW4_Results/MENG417_HW4_Submit_To_Professor.pdf")
assert pdf_path.is_file() and pdf_path.stat().st_size > 10000
print("PDF ready:", pdf_path, "|", pdf_path.stat().st_size, "bytes")
from google.colab import files
files.download(str(pdf_path))